In [1]:
# ============================================================
# Phase 1 (v2) | Step 1: FRED Refresh + GDP Extrapolation
# Refreshes market/global series to latest; extrapolates
# nominal USD GDP forward using RBI growth projection.
# Leaves RBI series + spliced CPI intact (Step 3 extends those).
# ============================================================

import pandas as pd
import numpy as np
from fredapi import Fred
import shutil

RAW       = '../data/raw/'
PROCESSED = '../data/processed/'

FRED_API_KEY = __import__('os').environ.get('FRED_API_KEY', '')
fred  = Fred(api_key=FRED_API_KEY)
START = '2000-01-01'
END   = '2026-06-01'                    # captures freshest available

# ── GDP extrapolation parameters (Option A) ───────────────────
RBI_REAL_GROWTH = 0.069     # RBI FY2026-27 real GDP projection
GDP_DEFLATOR    = 0.040     # inflation/deflator assumption (~4%)
NOMINAL_GROWTH  = (1 + RBI_REAL_GROWTH) * (1 + GDP_DEFLATOR) - 1
MONTHLY_FACTOR  = (1 + NOMINAL_GROWTH) ** (1/12)

def to_ms(s):
    s = s.resample('MS').last()
    s.index = s.index.to_period('M').to_timestamp()
    return s

# ── Backup current master ─────────────────────────────────────
shutil.copy(f'{PROCESSED}master.csv', f'{PROCESSED}master_backup_prelag.csv')
print("✓ Backed up master.csv → master_backup_prelag.csv\n")

master = pd.read_csv(f'{PROCESSED}master.csv', index_col=0, parse_dates=True)
master.index = pd.to_datetime(master.index).to_period('M').to_timestamp()
master = master[~master.index.duplicated(keep='last')]
master_idx = master.index

print("=" * 58)
print(f"STEP 1a: FRED refresh  (nominal growth = {NOMINAL_GROWTH*100:.2f}%)")
print("=" * 58)

# Refresh market/global series only — NOT cpi_india (preserve splice)
fred_map = {
    'CPIAUCSL'        : 'cpi_us',
    'FEDFUNDS'        : 'fed_funds_rate',
    'DTWEXBGS'        : 'dxy',
    'VIXCLS'          : 'vix',
    'GS10'            : 'us_10y_yield',
    'DCOILBRENTEU'    : 'brent',
    'WALCL'           : 'fed_balance_sheet',
    'IRSTCI01INM156N' : 'india_stir',
}
for sid, col in fred_map.items():
    try:
        s = to_ms(fred.get_series(sid, observation_start=START, observation_end=END))
        master[col] = s.reindex(master_idx)
        print(f"  ✓ {col:<22} ends {s.dropna().index[-1].strftime('%b %Y')}")
    except Exception as e:
        print(f"  ✗ {col:<22} FAILED — {e}")

# DXY backfill (2000–2006)
try:
    dxy_n = to_ms(fred.get_series('DTWEXM', observation_start=START,
                                   observation_end='2006-01-01'))
    master['dxy'] = master['dxy'].combine_first(dxy_n.reindex(master_idx))
except Exception:
    pass

# Re-patch repo rate gap with india_stir
gap = master['rbi_repo_rate'].isnull()
master.loc[gap, 'rbi_repo_rate'] = master.loc[gap, 'india_stir']
print(f"  ✓ rbi_repo_rate patched ({gap.sum()} months)")

print()
print("=" * 58)
print("STEP 1b: GDP extrapolation (RBI growth, FX-adjusted)")
print("=" * 58)

# Anchor on last genuine annual World Bank GDP point
wb_gdp = pd.read_csv(f'{RAW}wb_gdp_india.csv', index_col=0, parse_dates=True).squeeze()
wb_gdp.index = pd.to_datetime(wb_gdp.index).to_period('M').to_timestamp()
wb_gdp = wb_gdp[~wb_gdp.index.duplicated(keep='last')].dropna()

anchor_date    = wb_gdp.index.max()
anchor_year    = anchor_date.year
anchor_usd_gdp = float(wb_gdp.loc[anchor_date])

# Back out nominal INR GDP using avg INR/USD over the anchor year
yr_rates = master.loc[master.index.year == anchor_year, 'inr_usd'].dropna()
avg_rate = yr_rates.mean()
anchor_inr_gdp = anchor_usd_gdp * avg_rate

print(f"  Anchor: {anchor_year} nominal GDP = ${anchor_usd_gdp/1e12:.3f}T")
print(f"  Avg INR/USD {anchor_year}: {avg_rate:.2f}  →  "
      f"INR GDP ≈ ₹{anchor_inr_gdp/1e12:.1f}T (annual)")
print(f"  Extrapolating from Jan {anchor_year+1} at {NOMINAL_GROWTH*100:.2f}% nominal,")
print(f"  converted to USD at each month's actual INR/USD.\n")

anchor_ref = pd.Timestamp(year=anchor_year, month=12, day=1)
extrap = 0
for dt in master_idx:
    if dt <= anchor_ref:
        continue                          # keep genuine WB value
    rate = master.loc[dt, 'inr_usd']
    if pd.isnull(rate):
        continue                          # no FX yet (ragged edge) → leave as-is
    months = (dt.year - anchor_ref.year) * 12 + (dt.month - anchor_ref.month)
    inr_gdp_dt = anchor_inr_gdp * (MONTHLY_FACTOR ** months)
    master.loc[dt, 'gdp_india_usd'] = inr_gdp_dt / rate
    extrap += 1

print(f"  ✓ Extrapolated {extrap} months of gdp_india_usd")
# Show the FX effect at the edge
for d in [f'{anchor_year}-12-01', f'{anchor_year+1}-06-01', f'{anchor_year+1}-12-01']:
    dt = pd.Timestamp(d)
    if dt in master.index and pd.notna(master.loc[dt, 'gdp_india_usd']):
        print(f"    {d[:7]}  USD GDP=${master.loc[dt,'gdp_india_usd']/1e12:.3f}T  "
              f"@ INR {master.loc[dt,'inr_usd']:.2f}")

print()
print("=" * 58)
print("STEP 1c: Recompute affected derived series")
print("=" * 58)

master['cpi_us_yoy']       = master['cpi_us'].pct_change(12, fill_method=None) * 100
master['real_rate_us']     = master['fed_funds_rate'] - master['cpi_us_yoy']
master['real_rate_diff']   = master['real_rate_india'] - master['real_rate_us']
master['brent_yoy']        = master['brent'].pct_change(12, fill_method=None) * 100
master['fed_bs_yoy']       = master['fed_balance_sheet'].pct_change(12, fill_method=None) * 100
master['cab_pct_gdp']      = ((master['current_account'] * 4) /
                              (master['gdp_india_usd'] / 1e6)) * 100
master['remit_pct_gdp']    = (master['remittances_usd'] / master['gdp_india_usd']) * 100
print("  ✓ cpi_us_yoy, real_rate_diff, brent_yoy, fed_bs_yoy,")
print("    cab_pct_gdp, remit_pct_gdp recomputed")

print()
print("=" * 58)
print("VALIDATION")
print("=" * 58)
print(f"  Shape      : {master.shape}")
print(f"  Date range : {master.index[0].strftime('%b %Y')} → "
      f"{master.index[-1].strftime('%b %Y')}")
print()
print("  Edge checks:")
last = master.index[-1]
for col in ['dxy','brent','vix','fed_funds_rate','gdp_india_usd','cab_pct_gdp']:
    s = master[col].dropna()
    val = s.iloc[-1]
    disp = f"${val/1e12:.3f}T" if col == 'gdp_india_usd' else f"{val:.2f}"
    print(f"    {col:<18} {disp:>10}  (ends {s.index[-1].strftime('%b %Y')})")

master.to_csv(f'{PROCESSED}master.csv')
print(f"\n✓ master.csv saved  ({master.shape[0]} × {master.shape[1]})")
print("  Original preserved as master_backup_prelag.csv")

✓ Backed up master.csv → master_backup_prelag.csv

STEP 1a: FRED refresh  (nominal growth = 11.18%)
  ✓ cpi_us                 ends Apr 2026
  ✓ fed_funds_rate         ends May 2026
  ✓ dxy                    ends May 2026
  ✓ vix                    ends Jun 2026
  ✓ us_10y_yield           ends May 2026
  ✓ brent                  ends May 2026
  ✓ fed_balance_sheet      ends May 2026
  ✓ india_stir             ends Mar 2026
  ✓ rbi_repo_rate patched (1 months)

STEP 1b: GDP extrapolation (RBI growth, FX-adjusted)
  Anchor: 2024 nominal GDP = $3.910T
  Avg INR/USD 2024: 83.67  →  INR GDP ≈ ₹327.1T (annual)
  Extrapolating from Jan 2025 at 11.18% nominal,
  converted to USD at each month's actual INR/USD.

  ✓ Extrapolated 16 months of gdp_india_usd
    2024-12  USD GDP=$3.910T  @ INR 84.99
    2025-06  USD GDP=$4.015T  @ INR 85.90
    2025-12  USD GDP=$4.037T  @ INR 90.09

STEP 1c: Recompute affected derived series
  ✓ cpi_us_yoy, real_rate_diff, brent_yoy, fed_bs_yoy,
    cab_pct_gdp, 

In [2]:
import pandas as pd
m = pd.read_csv('../data/processed/master.csv', index_col=0, parse_dates=True)
m.index = pd.to_datetime(m.index)

print("BRENT — last 8 months:")
print(m['brent'].dropna().tail(8).round(2).to_string())
print("\nDXY (DTWEXBGS) — last 8 months:")
print(m['dxy'].dropna().tail(8).round(2).to_string())
print("\nReal end dates:")
for c in ['brent','dxy','vix','fed_funds_rate','india_stir']:
    print(f"  {c:<16} ends {m[c].dropna().index[-1].strftime('%b %Y')}")

BRENT — last 8 months:
2025-09-01     68.52
2025-10-01     65.44
2025-11-01     64.07
2025-12-01     61.35
2026-01-01     72.25
2026-02-01     71.32
2026-03-01    126.69
2026-04-01    124.24

DXY (DTWEXBGS) — last 8 months:
2025-09-01    120.14
2025-10-01    121.39
2025-11-01    121.05
2025-12-01    119.75
2026-01-01    117.90
2026-02-01    117.82
2026-03-01    121.04
2026-04-01    118.67

Real end dates:
  brent            ends Apr 2026
  dxy              ends Apr 2026
  vix              ends Apr 2026
  fed_funds_rate   ends Apr 2026
  india_stir       ends Mar 2026


In [3]:
import pandas as pd
import numpy as np
from fredapi import Fred

PROCESSED = '../data/processed/'
FRED_API_KEY = __import__('os').environ.get('FRED_API_KEY', '')
fred = Fred(api_key=FRED_API_KEY)

# Pull DAILY Brent to inspect Mar-Apr 2026
daily = fred.get_series('DCOILBRENTEU',
                        observation_start='2026-01-01',
                        observation_end='2026-06-01')
print("Brent — monthly AVERAGE vs LAST (2026):")
md = daily.resample('MS')
comp = pd.DataFrame({'mean': md.mean(), 'last': md.last(), 'max': md.max()})
print(comp.round(2).to_string())

Brent — monthly AVERAGE vs LAST (2026):
              mean    last     max
2026-01-01   66.60   72.25   72.25
2026-02-01   70.89   71.32   73.17
2026-03-01  103.13  126.69  126.69
2026-04-01  117.29  124.24  138.21
2026-05-01  109.39  102.75  118.26


In [4]:
import pandas as pd
from fredapi import Fred

PROCESSED = '../data/processed/'
FRED_API_KEY = __import__('os').environ.get('FRED_API_KEY', '')
fred = Fred(api_key=FRED_API_KEY)

master = pd.read_csv(f'{PROCESSED}master.csv', index_col=0, parse_dates=True)
master.index = pd.to_datetime(master.index).to_period('M').to_timestamp()

# Re-pull Brent as monthly AVERAGE (correct methodology for oil)
brent_daily = fred.get_series('DCOILBRENTEU',
                              observation_start='2000-01-01',
                              observation_end='2026-06-01')
brent_avg = brent_daily.resample('MS').mean()
brent_avg.index = brent_avg.index.to_period('M').to_timestamp()

master['brent'] = brent_avg.reindex(master.index)
master['brent_yoy'] = master['brent'].pct_change(12, fill_method=None) * 100

print("Brent (month-average) — last 6 months:")
print(master['brent'].dropna().tail(6).round(2).to_string())
print(f"\n  ends {master['brent'].dropna().index[-1].strftime('%b %Y')}")
print(f"  Brent normal (full-sample median): {master['brent'].median():.1f}")

master.to_csv(f'{PROCESSED}master.csv')
print("\n✓ Brent switched to month-average, master.csv updated")

Brent (month-average) — last 6 months:
2025-11-01     63.80
2025-12-01     62.54
2026-01-01     66.60
2026-02-01     70.89
2026-03-01    103.13
2026-04-01    117.29
Freq: MS

  ends Apr 2026
  Brent normal (full-sample median): 65.7

✓ Brent switched to month-average, master.csv updated


In [5]:
import pandas as pd
m = pd.read_csv('../data/processed/master.csv', index_col=0, parse_dates=True)
m.index = pd.to_datetime(m.index)
b = m['brent'].dropna()

print("BRENT NORMAL — candidate anchors")
print("─" * 48)
print(f"  Full-sample median (2000–2026) : {b.median():.1f}")
print(f"  Modeling window (2004–Jun25)   : {b['2004-04':'2025-06'].median():.1f}")
print(f"  Trailing 10yr median (120mo)   : {b.tail(120).median():.1f}")
print(f"  Trailing 5yr median (60mo)     : {b.tail(60).median():.1f}")
print(f"  Excl. war spike (drop Mar26+)  : {b[b.index < '2026-03-01'].median():.1f}")
print(f"  Prior value used (Phases 2-4)  : 72.3")
print()
print("  Last 12 months for context:")
print(b.tail(12).round(1).to_string())

BRENT NORMAL — candidate anchors
────────────────────────────────────────────────
  Full-sample median (2000–2026) : 65.7
  Modeling window (2004–Jun25)   : 72.7
  Trailing 10yr median (120mo)   : 69.9
  Trailing 5yr median (60mo)     : 80.2
  Excl. war spike (drop Mar26+)  : 65.4
  Prior value used (Phases 2-4)  : 72.3

  Last 12 months for context:
2025-05-01     64.5
2025-06-01     71.4
2025-07-01     71.0
2025-08-01     67.9
2025-09-01     68.0
2025-10-01     64.5
2025-11-01     63.8
2025-12-01     62.5
2026-01-01     66.6
2026-02-01     70.9
2026-03-01    103.1
2026-04-01    117.3


In [6]:
# brent_norm: structural long-run Brent anchor
# = full modeling-window median (2004-2025) = 72.7, prior value 72.3
# Deliberately fixed (not rolling) for cross-phase comparability and
# to exclude pre-2004 cheap-oil era + transient war/Ukraine spikes.
BRENT_NORM = 72.0

In [1]:
# ============================================================
# Phase 1 (v2) | Step 3a: Parse RBI files + extend FPI from BoP
# Builds: data/processed/step3a_raw.csv (verified raw series)
# Does NOT touch master.csv — that's Step 3b
# ============================================================
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, openpyxl
from datetime import datetime

RAW       = '../data/raw/'
PROCESSED = '../data/processed/'

MM = {'JAN':1,'FEB':2,'MAR':3,'APR':4,'MAY':5,'JUN':6,'JUL':7,'AUG':8,
      'SEP':9,'OCT':10,'NOV':11,'DEC':12,
      'JANUARY':1,'FEBRUARY':2,'MARCH':3,'APRIL':4,'JUNE':6,'JULY':7,
      'AUGUST':8,'SEPTEMBER':9,'OCTOBER':10,'NOVEMBER':11,'DECEMBER':12}
QM = {'Q1':4,'Q2':7,'Q3':10,'Q4':1}

def ms_index(d):
    s = pd.Series(d) if isinstance(d, dict) else d
    s.index = pd.to_datetime(s.index).to_period('M').to_timestamp()
    return s.sort_index()

# ── 1) INR/USD: date col 1, US$ avg col 5 ──────────────────────
wb = openpyxl.load_workbook(RAW+'rbi_inr_usd.xlsx', data_only=True); ws=wb.active
d = {}
for r in ws.iter_rows(min_row=9, values_only=True):
    dt, v = r[1], r[5]
    if isinstance(dt, datetime) and isinstance(v, (int,float)): d[dt] = float(v)
inr = ms_index(d)

# ── 2) REER (40-currency, 2015-16=100): date col 1, REER col 3 ─
wb = openpyxl.load_workbook(RAW+'rbi_reer.xlsx', data_only=True); ws=wb.active
d = {}
for r in ws.iter_rows(min_row=8, values_only=True):
    dt, v = r[1], r[3]
    if isinstance(dt, datetime) and isinstance(v, (int,float)): d[dt] = float(v)
reer = ms_index(d)

# ── 3) Reserves: Year col 1 (carry-fwd), Month col 2, total col 13 
wb = openpyxl.load_workbook(RAW+'rbi_reserves.xlsx', data_only=True); ws=wb.active
d = {}; cy = None
for r in ws.iter_rows(min_row=7, values_only=True):
    y = r[1]
    if y is not None:
        ys = str(y).strip()
        if ys.replace('.0','').isdigit(): cy = int(float(ys))
    m, tot = r[2], r[13]
    if cy and isinstance(m,str) and m.strip().upper() in MM and isinstance(tot,(int,float)):
        d[pd.Timestamp(year=cy, month=MM[m.strip().upper()], day=1)] = float(tot)
res = ms_index(d)

# ── 4) Trade: FY col 1, month col 2, exports col 3, imports col 4
wb = openpyxl.load_workbook(RAW+'rbi_trade.xlsx', data_only=True); ws=wb.active
rows = []
for r in ws.iter_rows(min_row=8, values_only=True):
    fy, m, ex, im = r[1], r[2], r[3], r[4]
    if (isinstance(fy,str) and '-' in fy and isinstance(m,str)
        and m.strip().upper() in MM
        and isinstance(ex,(int,float)) and isinstance(im,(int,float))):
        mo = MM[m.strip().upper()]
        ys = int(fy.split('-')[0]); yr = ys if mo >= 4 else ys + 1
        rows.append((pd.Timestamp(year=yr, month=mo, day=1), float(ex), float(im)))
tr = pd.DataFrame(rows, columns=['d','exports_usd_mn','imports_usd_mn']).set_index('d')
tr = tr[~tr.index.duplicated(keep='last')].sort_index()
tr['trade_balance_usd_mn'] = tr['exports_usd_mn'] - tr['imports_usd_mn']

# ── 5) CAB full BoP panel (Net rows; quarterly→quarter-start month)
wb = openpyxl.load_workbook(RAW+'rbi_cab.xlsx', data_only=True); ws=wb.active
COL = {'ca':5, 'merch':6, 'priv_transfer':20, 'capacct':24,
       'fdi_bop':26, 'portfolio_bop':35, 'loans':42,
       'banking':52, 'reserve_mov':63}
rec = {}
for r in ws.iter_rows(min_row=8, values_only=True):
    fy, q, tx = r[1], r[2], r[3]
    if (isinstance(fy,str) and '-' in fy and isinstance(q,str) and q.strip() in QM
        and isinstance(tx,str) and tx.strip().lower() == 'net'):
        ys = int(fy.split('-')[0]); mo = QM[q.strip()]
        yr = ys if mo >= 4 else ys + 1
        dt = pd.Timestamp(year=yr, month=mo, day=1)
        rec[dt] = {k: (float(r[i]) if isinstance(r[i],(int,float)) else np.nan)
                   for k, i in COL.items()}
cab = pd.DataFrame(rec).T.sort_index()

# ── 6) FPI new file: monthly BoP — [2] FDI, [20] Net Portfolio (USD mn)
wb = openpyxl.load_workbook(RAW+'rbi_fpi_flows.xlsx', data_only=True); ws=wb.active
d_pf = {}; d_fdi = {}
for r in ws.iter_rows(min_row=7, values_only=True):
    lbl = r[1]
    if isinstance(lbl,str) and ':' in lbl and '(' in lbl:
        try:
            yr = int(lbl.split(':')[0])
            mo = int(lbl.split(':')[1].split('(')[0])
            dt = pd.Timestamp(year=yr, month=mo, day=1)
            if isinstance(r[20],(int,float)): d_pf[dt]  = float(r[20])
            if isinstance(r[2], (int,float)): d_fdi[dt] = float(r[2])
        except: pass
new_portfolio = ms_index(d_pf)   # Mar 2011 → Mar 2026
new_fdi       = ms_index(d_fdi)

# ── 7) Splice fpi_flows_usd_mn at Mar 2011 ─────────────────────
# Pre-2011: preserve existing FII-definition values from master.csv
# Mar 2011 onward: new BoP Net Portfolio series (consistent with CAB)
old = pd.read_csv(PROCESSED+'master.csv', index_col=0, parse_dates=True)
old.index = pd.to_datetime(old.index).to_period('M').to_timestamp()
old_fpi   = old['fpi_flows_usd_mn'].dropna()

seam = pd.Timestamp('2011-03-01')
pre   = old_fpi[old_fpi.index < seam]
post  = new_portfolio[new_portfolio.index >= seam]
fpi_usd = pd.concat([pre, post]).sort_index()
fpi_usd = fpi_usd[~fpi_usd.index.duplicated(keep='last')]

# INR-crore from USD-mn × INR/USD ÷ 10
fpi_inr = (fpi_usd * inr.reindex(fpi_usd.index) / 10.0).dropna()

# ── 8) Assemble on monthly grid ─────────────────────────────────
idx = pd.date_range('2000-01-01','2026-05-01', freq='MS')
out = pd.DataFrame(index=idx); out.index.name = 'date'
out['inr_usd']              = inr.reindex(idx)
out['reer']                 = reer.reindex(idx)
out['reserves_usd_mn']      = res.reindex(idx)
out['exports_usd_mn']       = tr['exports_usd_mn'].reindex(idx)
out['imports_usd_mn']       = tr['imports_usd_mn'].reindex(idx)
out['trade_balance_usd_mn'] = tr['trade_balance_usd_mn'].reindex(idx)
for c in COL: out[c]        = cab[c].reindex(idx)    # quarterly: NaN except Apr/Jul/Oct/Jan
out['fpi_flows_usd_mn']     = fpi_usd.reindex(idx)
out['fpi_flows_inr_cr']     = fpi_inr.reindex(idx)
out['fdi_net_usd_mn']       = new_fdi.reindex(idx)

out.to_csv(PROCESSED+'step3a_raw.csv')

# ── 9) Inspection report ────────────────────────────────────────
print("="*68); print("STEP 3a — RAW SERIES BUILT"); print("="*68)
print(f"Output : {PROCESSED}step3a_raw.csv   shape={out.shape}")
print(f"Window : {idx[0].strftime('%b %Y')} → {idx[-1].strftime('%b %Y')}\n")

print("SERIES COVERAGE:")
print(f"  {'series':<24}{'first':>10}{'last':>10}{'n':>6}{'last value':>14}")
for c in out.columns:
    s = out[c].dropna()
    if not len(s): print(f"  {c:<24}  (empty)"); continue
    v = s.iloc[-1]
    disp = f"{v:.2f}" if abs(v)<1e4 else f"{v:.0f}"
    print(f"  {c:<24}{s.index[0].strftime('%b%Y'):>10}{s.index[-1].strftime('%b%Y'):>10}"
          f"{len(s):>6}{disp:>14}")

print("\nFPI SEAM (Feb 2011 = last old FII value; Mar 2011 onward = BoP Net Portfolio):")
for dt in pd.date_range('2010-12-01','2011-06-01', freq='MS'):
    v = out.loc[dt,'fpi_flows_usd_mn']
    if pd.notna(v): print(f"  {dt.strftime('%b %Y')}: ${v:>9.1f} mn")

print("\nCAB Q3 FY26 (Oct 2025) — full BoP panel, $mn:")
oct25 = out.loc[pd.Timestamp('2025-10-01')]
for c in COL: print(f"  {c:<18} {oct25[c]:>10.0f}")

print("\nFPI MONTHLY — last 12 months ($mn / INR cr / FDI $mn):")
print(f"  {'month':<10}{'FPI USD mn':>13}{'FPI INR cr':>14}{'FDI USD mn':>14}")
for dt in out.index[-14:-2]:
    u = out.loc[dt,'fpi_flows_usd_mn']
    r = out.loc[dt,'fpi_flows_inr_cr']
    f = out.loc[dt,'fdi_net_usd_mn']
    print(f"  {dt.strftime('%b %Y'):<10}{u:>13.1f}{r:>14.1f}{f:>14.1f}")

print("\n" + "="*68)
print("Step 3a complete. Inspect output, then proceed to Step 3b (master_v2.csv).")
print("="*68)

STEP 3a — RAW SERIES BUILT
Output : ../data/processed/step3a_raw.csv   shape=(317, 18)
Window : Jan 2000 → May 2026

SERIES COVERAGE:
  series                       first      last     n    last value
  inr_usd                    Jan2000   Apr2026   316         93.55
  reer                       Apr2004   Apr2026   265         90.96
  reserves_usd_mn            Jan2000   Feb2026   314        728494
  exports_usd_mn             Jan2000   Mar2026   315         38919
  imports_usd_mn             Jan2000   Mar2026   315         59589
  trade_balance_usd_mn       Jan2000   Mar2026   315        -20670
  ca                         Apr2000   Oct2025   103        -13198
  merch                      Apr2000   Oct2025   103        -93629
  priv_transfer              Apr2000   Oct2025   103         35367
  capacct                    Apr2000   Oct2025   103        -10005
  fdi_bop                    Apr2000   Oct2025   103      -3659.11
  portfolio_bop              Apr2000   Oct2025   103       -17

In [2]:
# ============================================================
# Phase 1 (v2) | Step 3b: Fold step3a_raw + Step 1 master
# → master_v2.csv (RBI refresh + BoP panel + recomputed derived)
# Preserves master.csv as Step 1 checkpoint.
# ============================================================
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, openpyxl, shutil
from datetime import datetime

RAW       = '../data/raw/'
PROCESSED = '../data/processed/'

# ── 1) Load Step 1's master + step3a_raw ────────────────────────
master = pd.read_csv(PROCESSED+'master.csv', index_col=0, parse_dates=True)
master.index = pd.to_datetime(master.index).to_period('M').to_timestamp()
master = master[~master.index.duplicated(keep='last')]

raw = pd.read_csv(PROCESSED+'step3a_raw.csv', index_col=0, parse_dates=True)
raw.index = pd.to_datetime(raw.index).to_period('M').to_timestamp()

idx = pd.date_range(min(master.index.min(), raw.index.min()),
                    max(master.index.max(), raw.index.max()), freq='MS')
master = master.reindex(idx)
raw    = raw.reindex(idx)

# ── 2) Pull NEER (trade-weighted, col 2) from rbi_reer.xlsx ────
wb = openpyxl.load_workbook(RAW+'rbi_reer.xlsx', data_only=True); ws=wb.active
d_neer = {}
for r in ws.iter_rows(min_row=8, values_only=True):
    dt, v = r[1], r[2]
    if isinstance(dt, datetime) and isinstance(v, (int,float)): d_neer[dt] = float(v)
neer = pd.Series(d_neer)
neer.index = pd.to_datetime(neer.index).to_period('M').to_timestamp()
neer = neer.sort_index().reindex(idx)

# ── 3) Overwrite columns from step3a ────────────────────────────
overwrite = {
    'inr_usd'           : 'inr_usd',
    'reer'              : 'reer',
    'fx_reserves_usd_mn': 'reserves_usd_mn',
    'current_account'   : 'ca',
    'exports_usd_mn'    : 'exports_usd_mn',
    'imports_usd_mn'    : 'imports_usd_mn',
    'trade_balance'     : 'trade_balance_usd_mn',
    'fpi_flows_usd_mn'  : 'fpi_flows_usd_mn',
    'fpi_flows_inr_cr'  : 'fpi_flows_inr_cr',
}
for mc, rc in overwrite.items(): master[mc] = raw[rc]
master['neer'] = neer

# ── 4) Add new BoP panel + monthly FDI columns ─────────────────
add = {
    'merch_balance_usd_mn'  : 'merch',
    'priv_transfer_usd_mn'  : 'priv_transfer',
    'capital_account_usd_mn': 'capacct',
    'fdi_bop_usd_mn'        : 'fdi_bop',
    'portfolio_bop_usd_mn'  : 'portfolio_bop',
    'loans_usd_mn'          : 'loans',
    'banking_usd_mn'        : 'banking',
    'reserve_mov_usd_mn'    : 'reserve_mov',
    'fdi_net_usd_mn'        : 'fdi_net_usd_mn',
}
for mc, rc in add.items(): master[mc] = raw[rc]

# ── 5) Recompute derived series ─────────────────────────────────
# NOTE: cpi_india / cpi_india_yoy preserved (Phase 1 splice ends Mar 2025)
master['cpi_us_yoy']             = master['cpi_us'].pct_change(12, fill_method=None) * 100
master['inflation_diff']         = master['cpi_india_yoy'] - master['cpi_us_yoy']
master['real_rate_india']        = master['rbi_repo_rate'] - master['cpi_india_yoy']
master['real_rate_us']           = master['fed_funds_rate'] - master['cpi_us_yoy']
master['real_rate_diff']         = master['real_rate_india'] - master['real_rate_us']
master['justified_depreciation'] = master['inflation_diff']
master['brent_yoy']              = master['brent'].pct_change(12, fill_method=None) * 100
master['fed_bs_yoy']             = master['fed_balance_sheet'].pct_change(12, fill_method=None) * 100
master['inr_yoy_chg']            = master['inr_usd'].pct_change(12, fill_method=None) * 100
master['reserves_mom_chg']       = master['fx_reserves_usd_mn'].diff()
master['import_cover']           = master['fx_reserves_usd_mn'] / master['imports_usd_mn']
master['cab_pct_gdp']            = (master['current_account']*4) / (master['gdp_india_usd']/1e6) * 100
master['remit_pct_gdp']          = master['remittances_usd'] / master['gdp_india_usd'] * 100
master['fpi_pct_gdp']            = (master['fpi_flows_usd_mn']*12) / (master['gdp_india_usd']/1e6) * 100

# ── 6) Trim purely-empty trailing rows ─────────────────────────
nonempty = master.dropna(how='all')
master = master.loc[:nonempty.index.max()]

# ── 7) Backup + save ────────────────────────────────────────────
shutil.copy(PROCESSED+'master.csv', PROCESSED+'master_v1_post_step1.csv')
master.to_csv(PROCESSED+'master_v2.csv')

# ── 8) Validation report ────────────────────────────────────────
print("="*70); print("STEP 3b — MASTER_V2 BUILT"); print("="*70)
print(f"Output : {PROCESSED}master_v2.csv")
print(f"Backup : master.csv → master_v1_post_step1.csv")
print(f"Shape  : {master.shape}")
print(f"Window : {master.index[0].strftime('%b %Y')} → {master.index[-1].strftime('%b %Y')}\n")

print("SERIES EDGE MAP:")
groups = {
    'Market/global (FRED)' : ['cpi_us','fed_funds_rate','dxy','vix','us_10y_yield','brent','fed_balance_sheet'],
    'India market'         : ['inr_usd','reer','neer','fx_reserves_usd_mn','india_stir','rbi_repo_rate'],
    'India macro (annual)' : ['gdp_india_usd','remittances_usd'],
    'India prices'         : ['cpi_india','cpi_india_yoy'],
    'Trade (monthly)'      : ['exports_usd_mn','imports_usd_mn','trade_balance'],
    'BoP panel (quarterly)': ['current_account','merch_balance_usd_mn','priv_transfer_usd_mn',
                              'capital_account_usd_mn','fdi_bop_usd_mn','portfolio_bop_usd_mn',
                              'loans_usd_mn','banking_usd_mn','reserve_mov_usd_mn'],
    'Flows (monthly)'      : ['fpi_flows_usd_mn','fpi_flows_inr_cr','fdi_net_usd_mn'],
    'Derived'              : ['real_rate_diff','cab_pct_gdp','fpi_pct_gdp','import_cover','brent_yoy'],
}
for grp, cols in groups.items():
    print(f"\n  {grp}:")
    for c in cols:
        if c in master.columns:
            s = master[c].dropna()
            if len(s):
                v=s.iloc[-1]; disp=f"{v:.2f}" if abs(v)<1e4 else f"{v:.0f}"
                print(f"    {c:<26} ends {s.index[-1].strftime('%b %Y')}  last={disp}")
            else:
                print(f"    {c:<26} EMPTY")

# Sanity checks
print("\n" + "="*70); print("SANITY CHECKS"); print("="*70)
checks = []
oct25 = master.loc[pd.Timestamp('2025-10-01')]
checks.append(("CAB Q3 FY26: current_account=-13198",      abs(oct25['current_account'] - (-13198)) < 1))
checks.append(("CAB Q3 FY26: capital_account=-10005",      abs(oct25['capital_account_usd_mn'] - (-10005)) < 1))
checks.append(("CAB Q3 FY26: fdi_bop NEGATIVE (capital flight)",  oct25['fdi_bop_usd_mn'] < 0))
checks.append(("CAB Q3 FY26: reserve_mov=+24409 ($24bn drawdown)", abs(oct25['reserve_mov_usd_mn'] - 24409) < 1))
mar26 = master.loc[pd.Timestamp('2026-03-01')]
checks.append(("FPI Mar 2026 ≈ -$13,343mn (oil shock)",    abs(mar26['fpi_flows_usd_mn'] - (-13343)) < 10))
feb11 = master.loc[pd.Timestamp('2011-02-01'),'fpi_flows_usd_mn']
mar11 = master.loc[pd.Timestamp('2011-03-01'),'fpi_flows_usd_mn']
checks.append((f"FPI seam smooth: Feb11={feb11:.0f}→Mar11={mar11:.0f}", abs(mar11-feb11) < 2000))
checks.append(("INR Apr 2026 ≈ 93.55",
               abs(master.loc[pd.Timestamp('2026-04-01'),'inr_usd'] - 93.55) < 0.5))
checks.append(("TWIN DEFICIT: CAD<0 AND capacct<0 in Q3 FY26",
               oct25['current_account']<0 and oct25['capital_account_usd_mn']<0))
for label, ok in checks:
    print(f"  {'✓' if ok else '✗'}  {label}")

# Recent ragged edge
print("\n" + "="*70); print("RECENT EDGE — last 6 months (ragged is correct)"); print("="*70)
cols_view = ['inr_usd','reer','dxy','brent','fx_reserves_usd_mn',
             'fpi_flows_usd_mn','fdi_net_usd_mn','current_account','fpi_pct_gdp']
for dt, row in master[cols_view].tail(6).iterrows():
    print(f"\n  {dt.strftime('%b %Y')}:")
    for c in cols_view:
        v=row[c]
        if pd.isna(v): print(f"    {c:<22} ·")
        else:
            disp=f"{v:.2f}" if abs(v)<1e4 else f"{v:.0f}"
            print(f"    {c:<22} {disp}")

STEP 3b — MASTER_V2 BUILT
Output : ../data/processed/master_v2.csv
Backup : master.csv → master_v1_post_step1.csv
Shape  : (316, 52)
Window : Jan 2000 → Apr 2026

SERIES EDGE MAP:

  Market/global (FRED):
    cpi_us                     ends Apr 2026  last=332.41
    fed_funds_rate             ends Apr 2026  last=3.64
    dxy                        ends Apr 2026  last=118.67
    vix                        ends Apr 2026  last=16.89
    us_10y_yield               ends Apr 2026  last=4.32
    brent                      ends Apr 2026  last=117.29
    fed_balance_sheet          ends Apr 2026  last=6699950

  India market:
    inr_usd                    ends Apr 2026  last=93.55
    reer                       ends Apr 2026  last=90.96
    neer                       ends Apr 2026  last=79.66
    fx_reserves_usd_mn         ends Feb 2026  last=728494
    india_stir                 ends Mar 2026  last=5.50
    rbi_repo_rate              ends Mar 2026  last=5.50

  India macro (annual):
    gdp_in

In [3]:
# ============================================================
# Phase 1 (v2) | Step 4: Apply point-in-time publication lags
# Strict mode: every input gets its publication delay applied,
# then derived series are recomputed from lagged inputs.
# Output: master_v2_lagged.csv  (master_v2.csv preserved unlagged)
# ============================================================
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np

PROCESSED = '../data/processed/'

# Load unlagged master_v2
m = pd.read_csv(PROCESSED+'master_v2.csv', index_col=0, parse_dates=True)
m.index = pd.to_datetime(m.index).to_period('M').to_timestamp()

# ── Publication lag schedule (months to shift forward) ────────
# Series not listed are real-time (lag = 0): inr_usd, reer, neer, dxy, vix,
# fed_funds_rate, us_10y_yield, brent, fed_balance_sheet, fx_reserves_usd_mn,
# rbi_repo_rate, india_stir
LAG = {
    'cpi_us': 1,                                              # ~2wk BLS lag
    'cpi_india': 1,                                           # MOSPI ~12 days
    'exports_usd_mn': 1, 'imports_usd_mn': 1, 'trade_balance': 1,  # Commerce ~2wk
    'fpi_flows_usd_mn': 2, 'fpi_flows_inr_cr': 2,             # RBI Bulletin ~6-8wk
    'fdi_net_usd_mn': 2,
    'current_account': 6, 'merch_balance_usd_mn': 6,          # BoP quarterly:
    'priv_transfer_usd_mn': 6, 'capital_account_usd_mn': 6,   # marker at Q-start +
    'fdi_bop_usd_mn': 6, 'portfolio_bop_usd_mn': 6,           # 3mo quarter +
    'loans_usd_mn': 6, 'banking_usd_mn': 6,                   # 3mo publication
    'reserve_mov_usd_mn': 6,                                  # = 6mo from marker
    'gdp_india_usd': 3, 'gdp_us_usd': 3,                      # annual, strict
    'gdp_pc_india': 3, 'gdp_pc_us': 3,
    'remittances_usd': 3,
}

# Extend index by max lag so shifted tail values aren't lost
max_lag = max(LAG.values())
new_idx = pd.date_range(m.index[0], m.index[-1] + pd.DateOffset(months=max_lag), freq='MS')
m = m.reindex(new_idx)

# ── Apply lags to RAW inputs only ───────────────────────────────
ml = m.copy()
for col, k in LAG.items():
    if col in ml.columns: ml[col] = m[col].shift(k)

# ── Recompute ALL derived series from LAGGED inputs ────────────
ml['cpi_us_yoy']             = ml['cpi_us'].pct_change(12, fill_method=None) * 100
ml['cpi_india_yoy']          = ml['cpi_india'].pct_change(12, fill_method=None) * 100
ml['inflation_diff']         = ml['cpi_india_yoy'] - ml['cpi_us_yoy']
ml['real_rate_india']        = ml['rbi_repo_rate'] - ml['cpi_india_yoy']
ml['real_rate_us']           = ml['fed_funds_rate'] - ml['cpi_us_yoy']
ml['real_rate_diff']         = ml['real_rate_india'] - ml['real_rate_us']
ml['justified_depreciation'] = ml['inflation_diff']
ml['brent_yoy']              = ml['brent'].pct_change(12, fill_method=None) * 100
ml['fed_bs_yoy']             = ml['fed_balance_sheet'].pct_change(12, fill_method=None) * 100
ml['inr_yoy_chg']            = ml['inr_usd'].pct_change(12, fill_method=None) * 100
ml['reserves_mom_chg']       = ml['fx_reserves_usd_mn'].diff()
ml['import_cover']           = ml['fx_reserves_usd_mn'] / ml['imports_usd_mn']
ml['cab_pct_gdp']            = (ml['current_account']*4) / (ml['gdp_india_usd']/1e6) * 100
ml['remit_pct_gdp']          = ml['remittances_usd'] / ml['gdp_india_usd'] * 100
ml['fpi_pct_gdp']            = (ml['fpi_flows_usd_mn']*12) / (ml['gdp_india_usd']/1e6) * 100

# Trim purely-empty trailing rows
ne = ml.dropna(how='all')
ml = ml.loc[:ne.index.max()]

# Save (unlagged master_v2.csv stays as reference)
ml.to_csv(PROCESSED+'master_v2_lagged.csv')

# ── Validation report ───────────────────────────────────────────
print("="*70); print("STEP 4 — POINT-IN-TIME LAGS APPLIED"); print("="*70)
print(f"Output : {PROCESSED}master_v2_lagged.csv")
print(f"Preserved: master_v2.csv (unlagged reference)")
print(f"Shape  : {ml.shape}")
print(f"Window : {ml.index[0].strftime('%b %Y')} → {ml.index[-1].strftime('%b %Y')}\n")

# Critical lag checks
print("LAG VERIFICATION (known values land at expected dates):")
checks = []
# Q3 FY26 BoP: Oct 2025 marker + 6mo = Apr 2026
v = ml.loc[pd.Timestamp('2026-04-01'),'current_account']
checks.append((f"Q3 FY26 current_account (-13198) at Apr 2026: {v:.0f}", abs(v-(-13198))<1))
# Mar 2026 FPI: + 2mo = May 2026
v = ml.loc[pd.Timestamp('2026-05-01'),'fpi_flows_usd_mn']
checks.append((f"Mar 2026 FPI (-13343) at May 2026: {v:.0f}", abs(v-(-13343))<10))
# Real-time series unchanged
v = ml.loc[pd.Timestamp('2026-04-01'),'inr_usd']
checks.append((f"INR Apr 2026 unchanged at 93.55: {v:.2f}", abs(v-93.55)<0.5))
for label, ok in checks:
    print(f"  {'✓' if ok else '✗'}  {label}")

print("\nEDGE COMPARISON (unlagged → lagged ends):")
groups = {
    'Real-time (lag=0)'     : ['inr_usd','dxy','brent','fx_reserves_usd_mn','fed_funds_rate'],
    'CPI/Trade (lag=1)'     : ['cpi_us','cpi_india','exports_usd_mn','imports_usd_mn'],
    'FPI/FDI monthly (lag=2)': ['fpi_flows_usd_mn','fdi_net_usd_mn'],
    'GDP annual (lag=3)'    : ['gdp_india_usd','remittances_usd'],
    'BoP quarterly (lag=6)' : ['current_account','capital_account_usd_mn','fdi_bop_usd_mn'],
    'Derived (downstream)'  : ['real_rate_diff','cab_pct_gdp','fpi_pct_gdp','import_cover'],
}
for grp, cols in groups.items():
    print(f"\n  {grp}:")
    for c in cols:
        if c in ml.columns:
            u = m[c].dropna(); l = ml[c].dropna()
            if len(u) and len(l):
                u_end = u.index[-1].strftime('%b %Y')
                l_end = l.index[-1].strftime('%b %Y')
                print(f"    {c:<24} {u_end} → {l_end}")

# Live edge of LAGGED master
print("\n" + "="*70)
print("LAGGED MASTER — last 8 months (this is what the model sees in real time)")
print("="*70)
cv = ['inr_usd','brent','cpi_india_yoy','real_rate_diff','fpi_flows_usd_mn',
      'current_account','cab_pct_gdp','fpi_pct_gdp']
for dt, row in ml[cv].tail(8).iterrows():
    print(f"\n  {dt.strftime('%b %Y')}:")
    for c in cv:
        v=row[c]
        if pd.isna(v): print(f"    {c:<22} ·")
        else:
            disp = f"{v:.2f}" if abs(v)<1e4 else f"{v:.0f}"
            print(f"    {c:<22} {disp}")

STEP 4 — POINT-IN-TIME LAGS APPLIED
Output : ../data/processed/master_v2_lagged.csv
Preserved: master_v2.csv (unlagged reference)
Shape  : (319, 52)
Window : Jan 2000 → Jul 2026

LAG VERIFICATION (known values land at expected dates):
  ✓  Q3 FY26 current_account (-13198) at Apr 2026: -13198
  ✓  Mar 2026 FPI (-13343) at May 2026: -13343
  ✓  INR Apr 2026 unchanged at 93.55: 93.55

EDGE COMPARISON (unlagged → lagged ends):

  Real-time (lag=0):
    inr_usd                  Apr 2026 → Apr 2026
    dxy                      Apr 2026 → Apr 2026
    brent                    Apr 2026 → Apr 2026
    fx_reserves_usd_mn       Feb 2026 → Feb 2026
    fed_funds_rate           Apr 2026 → Apr 2026

  CPI/Trade (lag=1):
    cpi_us                   Apr 2026 → May 2026
    cpi_india                Apr 2026 → May 2026
    exports_usd_mn           Mar 2026 → Apr 2026
    imports_usd_mn           Mar 2026 → Apr 2026

  FPI/FDI monthly (lag=2):
    fpi_flows_usd_mn         Mar 2026 → May 2026
    fdi_net

In [4]:
# ============================================================
# Phase 2 (v2): REER Structural Anchor
# Input: master_v2_lagged.csv
# Output: phase2_equilibrium_v2.csv
# REER is lag=0 (real-time), so lagged=unlagged for this phase.
# Uses 40-currency REER, base 2015-16=100 (upgrade from 36-cur)
# ============================================================
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
from statsmodels.tsa.filters.hp_filter import hpfilter

PROCESSED = '../data/processed/'

m = pd.read_csv(PROCESSED+'master_v2_lagged.csv', index_col=0, parse_dates=True)
m.index = pd.to_datetime(m.index).to_period('M').to_timestamp()

# ── Align REER + INR/USD on common window ───────────────────────
reer = m['reer'].dropna()
inr  = m['inr_usd'].reindex(reer.index).dropna()
common = reer.index.intersection(inr.index)
reer, inr = reer.loc[common], inr.loc[common]

print("="*70)
print("PHASE 2 — REER STRUCTURAL ANCHOR (40-currency, 2015-16=100)")
print("="*70)
print(f"Window: {reer.index[0].strftime('%b %Y')} → {reer.index[-1].strftime('%b %Y')}  (n={len(reer)})")

# ── HP filter (monthly λ=129,600) ──────────────────────────────
cycle, trend = hpfilter(reer.values, lamb=129600)
reer_trend = pd.Series(trend, index=reer.index)

# ── Misalignment: positive = INR undervalued / too weak ────────
# REER < trend → reer_trend/reer > 1 → misalign > 0 → undervalued
misalign = (reer_trend / reer - 1) * 100

# ── Fair INR/USD: where INR would be if REER = trend ───────────
# REER < trend → reer/reer_trend < 1 → fair < actual → fair is stronger
inr_fair = inr * (reer / reer_trend)

# ── Build + save output ────────────────────────────────────────
out = pd.DataFrame({
    'inr_usd': inr,
    'reer': reer,
    'reer_trend': reer_trend,
    'reer_deviation_pct': (reer / reer_trend - 1) * 100,
    'misalignment_pct': misalign,
    'inr_fair_reer': inr_fair,
}, index=common)
out.to_csv(PROCESSED+'phase2_equilibrium_v2.csv')

# ── Episode validation ──────────────────────────────────────────
episodes = {
    'GFC (Oct 2008)'            : '2008-10-01',
    'Taper tantrum (Aug 2013)'  : '2013-08-01',
    'COVID (Apr 2020)'          : '2020-04-01',
    'Fed tightening (Oct 2022)' : '2022-10-01',
    'Pre-war (Dec 2025)'        : '2025-12-01',
    'Oil shock (Mar 2026)'      : '2026-03-01',
    'Current (Apr 2026)'        : '2026-04-01',
}
print(f"\nEPISODE VALIDATION:")
print(f"  {'Episode':<28}{'REER':>8}{'Trend':>8}{'Misalign%':>10}{'INR act':>9}{'INR fair':>9}")
for label, dt_str in episodes.items():
    dt = pd.Timestamp(dt_str)
    if dt in out.index:
        r = out.loc[dt]
        print(f"  {label:<28}{r['reer']:>8.1f}{r['reer_trend']:>8.1f}"
              f"{r['misalignment_pct']:>10.1f}{r['inr_usd']:>9.2f}{r['inr_fair_reer']:>9.2f}")

# ── Summary ─────────────────────────────────────────────────────
print(f"\nSUMMARY:")
print(f"  REER range      : {reer.min():.1f} – {reer.max():.1f}")
print(f"  Trend range     : {reer_trend.min():.1f} – {reer_trend.max():.1f}")
print(f"  Current misalign: {misalign.iloc[-1]:+.1f}%"
      f" ({'INR undervalued' if misalign.iloc[-1]>0 else 'INR overvalued'})")
print(f"  Fair INR/USD    : {inr_fair.iloc[-1]:.2f} vs actual {inr.iloc[-1]:.2f}"
      f" (gap: {inr.iloc[-1] - inr_fair.iloc[-1]:.2f})")
print(f"\n  ← Prior Phase 2 (Jun 2025): +4.1%, fair ~82.5")
print(f"  → Current (Apr 2026):       {misalign.iloc[-1]:+.1f}%, fair {inr_fair.iloc[-1]:.2f}")
print(f"     Change reflects: oil-shock REER depreciation + HP trend adaptation")

print(f"\n✓ Saved: {PROCESSED}phase2_equilibrium_v2.csv  ({out.shape[0]} × {out.shape[1]})")

PHASE 2 — REER STRUCTURAL ANCHOR (40-currency, 2015-16=100)
Window: Apr 2004 → Apr 2026  (n=265)

EPISODE VALIDATION:
  Episode                         REER   Trend Misalign%  INR act INR fair
  GFC (Oct 2008)                  87.6    93.6       6.9    48.64    45.52
  Taper tantrum (Aug 2013)        88.7    96.5       8.8    63.21    58.12
  COVID (Apr 2020)               102.4   103.9       1.4    76.24    75.17
  Fed tightening (Oct 2022)      104.2   103.8      -0.3    82.34    82.59
  Pre-war (Dec 2025)              95.1    99.6       4.7    90.09    86.06
  Oil shock (Mar 2026)            92.6    99.0       7.0    92.76    86.69
  Current (Apr 2026)              91.0    98.9       8.7    93.55    86.06

SUMMARY:
  REER range      : 86.2 – 108.0
  Trend range     : 89.1 – 104.1
  Current misalign: +8.7% (INR undervalued)
  Fair INR/USD    : 86.06 vs actual 93.55 (gap: 7.49)

  ← Prior Phase 2 (Jun 2025): +4.1%, fair ~82.5
  → Current (Apr 2026):       +8.7%, fair 86.06
     Change

In [1]:
# ============================================================
# Phase 3 (v2): FEER Sustainability Engine + Conditional Norm
# Input: master_v2.csv (UNLAGGED — contemporaneous alignment)
#        phase2_equilibrium_v2.csv (REER trend)
# Output: phase3_feer_v2.csv (quarterly, with financing decomposition)
# ============================================================
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
import statsmodels.api as sm

PROCESSED = '../data/processed/'

# ── Locked parameters ───────────────────────────────────────────
REER_SEMI       = -0.267    # IMF-EBA validated (−0.2 to −0.4)
OIL_ELAST       = -2.45     # prior Phase 3, confirmed by re-estimation (−2.55)
BRENT_NORM      = 72.0      # structural median, fixed
CAD_NORM_STATIC = -2.5      # static target

# Use UNLAGGED master for internal computation (contemporaneous alignment)
m = pd.read_csv(PROCESSED+'master_v2.csv', index_col=0, parse_dates=True)
m.index = pd.to_datetime(m.index).to_period('M').to_timestamp()

p2 = pd.read_csv(PROCESSED+'phase2_equilibrium_v2.csv', index_col=0, parse_dates=True)
p2.index = pd.to_datetime(p2.index).to_period('M').to_timestamp()

# ── Build quarterly contemporaneous dataset ─────────────────────
q_markers = m['current_account'].dropna().index
recs = []
for dt in q_markers:
    qm = pd.date_range(dt, periods=3, freq='MS')     # 3 months in quarter
    brent_q = m.loc[m.index.isin(qm), 'brent'].mean()
    reer_q  = m.loc[m.index.isin(qm), 'reer'].mean()
    inr_q   = m.loc[m.index.isin(qm), 'inr_usd'].mean()
    rtrend  = p2['reer_trend'].reindex(qm).mean()

    ca   = m.loc[dt,'current_account']
    gdp  = m.loc[dt,'gdp_india_usd']
    fdi  = m.loc[dt,'fdi_bop_usd_mn']
    loans_v = m.loc[dt,'loans_usd_mn']

    if pd.notna(gdp) and gdp > 0 and pd.notna(ca) and pd.notna(brent_q):
        recs.append({
            'date': dt,
            'cad_pct': (ca * 4) / (gdp / 1e6) * 100,
            'brent_q': brent_q, 'reer_q': reer_q, 'inr_q': inr_q,
            'reer_trend_q': rtrend, 'fdi': fdi, 'loans': loans_v, 'gdp': gdp,
        })
q = pd.DataFrame(recs).set_index('date').sort_index()
q['log_brent'] = np.log(q['brent_q'])

print("="*70)
print("PHASE 3 — FEER SUSTAINABILITY + CONDITIONAL NORM")
print("="*70)
print(f"Quarterly dataset: {q.index[0].strftime('%b %Y')} → {q.index[-1].strftime('%b %Y')}  (n={len(q)})")
print(f"Locked: reer_semi={REER_SEMI}, oil_elast={OIL_ELAST}, brent_norm=${BRENT_NORM}")

# ── Re-estimate oil elasticity as confirmation ──────────────────
X = sm.add_constant(q['log_brent']); y = q['cad_pct']
res = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags':4})
print(f"\nOil elasticity re-estimated: {res.params['log_brent']:.2f} "
      f"(p={res.pvalues['log_brent']:.4f}, R²={res.rsquared:.3f})")
print(f"  Locked at {OIL_ELAST} (prior, within confidence band)")

# ── FEER computation ────────────────────────────────────────────
q['oil_drag']        = OIL_ELAST * (q['log_brent'] - np.log(BRENT_NORM))
q['cad_underlying']  = q['cad_pct'] - q['oil_drag']

# Static FEER
q['cad_gap_static']  = q['cad_underlying'] - CAD_NORM_STATIC
q['feer_static_pct'] = -q['cad_gap_static'] / REER_SEMI

# Conditional FEER (stable financing = FDI + loans, capital account only)
q['stable_fin']      = q['fdi'] + q['loans']
q['stable_pct_gdp']  = (q['stable_fin'] * 4) / (q['gdp'] / 1e6) * 100
q['cad_norm_cond']   = np.where(q['stable_fin']>0, -q['stable_pct_gdp'], 0)
q['cad_gap_cond']    = q['cad_underlying'] - q['cad_norm_cond']
q['feer_cond_pct']   = -q['cad_gap_cond'] / REER_SEMI

# INR fair values
q['inr_fair_static'] = q['inr_q'] / (1 + q['feer_static_pct']/100)
q['inr_fair_cond']   = q['inr_q'] / (1 + q['feer_cond_pct']/100)

# Financing premium
q['financing_premium'] = q['feer_static_pct'] - q['feer_cond_pct']

# Save
q.to_csv(PROCESSED+'phase3_feer_v2.csv')

# ── Episode validation ──────────────────────────────────────────
episodes = {
    'GFC (Oct 2008)'        : '2008-10-01',
    'Taper (Jul 2013)'      : '2013-07-01',
    'Post-taper (Jan 2014)' : '2014-01-01',
    'COVID (Apr 2020)'      : '2020-04-01',
    'Fed tight (Oct 2022)'  : '2022-10-01',
    'Pre-war (Jul 2025)'    : '2025-07-01',
    'Q3 FY26 (Oct 2025)'    : '2025-10-01',
}
print(f"\nEPISODE VALIDATION:")
print(f"  {'Episode':<24}{'CAD%':>7}{'Brent':>7}{'OilDrg':>7}{'Undly':>7}"
      f"{'Static':>8}{'Cond':>8}{'CondNrm':>8}{'FinPrm':>8}")
for lbl, dt_str in episodes.items():
    dt = pd.Timestamp(dt_str)
    if dt in q.index:
        r = q.loc[dt]
        print(f"  {lbl:<24}{r['cad_pct']:>7.1f}{r['brent_q']:>7.0f}{r['oil_drag']:>7.2f}"
              f"{r['cad_underlying']:>7.1f}{r['feer_static_pct']:>8.1f}{r['feer_cond_pct']:>8.1f}"
              f"{r['cad_norm_cond']:>8.1f}{r['financing_premium']:>8.1f}")

# ── Latest quarter deep dive ────────────────────────────────────
print("\n" + "="*70)
print("LATEST QUARTER: Q3 FY26 (Oct-Dec 2025)")
print("="*70)
last = q.iloc[-1]
print(f"  Quarter Brent avg  : ${last['brent_q']:.0f}  (normal ${BRENT_NORM})")
print(f"  Actual CAD         : {last['cad_pct']:.1f}% of GDP")
print(f"  Oil drag           : {last['oil_drag']:+.2f}pp")
print(f"  Underlying CAD     : {last['cad_underlying']:.1f}%\n")
print(f"  STATIC FEER (norm={CAD_NORM_STATIC}%):")
print(f"    Misalignment     : {last['feer_static_pct']:+.1f}%"
      f" ({'INR undervalued' if last['feer_static_pct']>0 else 'INR overvalued'})")
print(f"    Fair INR/USD     : {last['inr_fair_static']:.2f}\n")
print(f"  CONDITIONAL FEER (financing-adjusted):")
print(f"    Stable financing : FDI={last['fdi']:.0f} + Loans={last['loans']:.0f}"
      f" = {last['stable_fin']:.0f} $mn/qtr")
print(f"    Conditional norm : {last['cad_norm_cond']:.1f}%")
print(f"    Misalignment     : {last['feer_cond_pct']:+.1f}%"
      f" ({'INR undervalued' if last['feer_cond_pct']>0 else 'INR overvalued'})")
print(f"    Fair INR/USD     : {last['inr_fair_cond']:.2f}\n")
print(f"  FINANCING PREMIUM  : {last['financing_premium']:+.1f}pp")
print(f"  (gap = how much capital-account stress distorts the fair value)")

# ── Cross-model comparison ──────────────────────────────────────
print(f"\n  ← Prior Phase 3 (Jun 2025): static +7.4%, fair ~82.0")
print(f"  → Current Q3 FY26:          static {last['feer_static_pct']:+.1f}%, fair {last['inr_fair_static']:.2f}")
print(f"                               conditional {last['feer_cond_pct']:+.1f}%, fair {last['inr_fair_cond']:.2f}")

print(f"\n✓ Saved: {PROCESSED}phase3_feer_v2.csv  ({q.shape[0]} × {q.shape[1]})")

PHASE 3 — FEER SUSTAINABILITY + CONDITIONAL NORM
Quarterly dataset: Apr 2000 → Oct 2025  (n=103)
Locked: reer_semi=-0.267, oil_elast=-2.45, brent_norm=$72.0

Oil elasticity re-estimated: -2.61 (p=0.0000, R²=0.398)
  Locked at -2.45 (prior, within confidence band)

EPISODE VALIDATION:
  Episode                    CAD%  Brent OilDrg  Undly  Static    Cond CondNrm  FinPrm
  GFC (Oct 2008)             -4.0     55   0.68   -4.6    -8.0   -13.2    -1.1     5.2
  Taper (Jul 2013)           -1.1    110  -1.04   -0.1     9.1     5.9    -1.6     3.2
  Post-taper (Jan 2014)      -0.3    108  -1.00    0.7    12.1     4.6    -0.5     7.5
  COVID (Apr 2020)            2.8     29   2.20    0.7    11.8     3.7    -0.3     8.1
  Fed tight (Oct 2022)       -2.0     89  -0.51   -1.5     3.7    -4.5    -0.3     8.2
  Pre-war (Jul 2025)         -1.4     69   0.11   -1.5     3.7    -3.3    -0.6     7.0
  Q3 FY26 (Oct 2025)         -1.3     64   0.30   -1.6     3.3    -2.4    -1.0     5.8

LATEST QUARTER: Q3

In [2]:
# ============================================================
# Phase 4 (v2): BEER Equilibrium + Regime Engine
# Input: master_v2_lagged.csv (point-in-time)
# Output: phase4_beer_v2.csv, phase4_regimes_v2.csv, beer_coefs_v2.json
# ============================================================
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, json
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression

PROCESSED = '../data/processed/'

ml = pd.read_csv(PROCESSED+'master_v2_lagged.csv', index_col=0, parse_dates=True)
ml.index = pd.to_datetime(ml.index).to_period('M').to_timestamp()

# ── Align BEER variables (dropna on all) ────────────────────────
beer_cols = ['inr_usd','dxy','real_rate_diff','fpi_pct_gdp','brent','vix']
d = ml[beer_cols].dropna().copy()
d['log_inr']   = np.log(d['inr_usd'])
d['log_dxy']   = np.log(d['dxy'])
d['log_brent'] = np.log(d['brent'])

print("="*70)
print("PHASE 4 — BEER EQUILIBRIUM + REGIME ENGINE")
print("="*70)
print(f"Sample: {d.index[0].strftime('%b %Y')} → {d.index[-1].strftime('%b %Y')}  (n={len(d)})")

# ── 1) Full BEER regression ────────────────────────────────────
X = sm.add_constant(d[['log_dxy','real_rate_diff','fpi_pct_gdp','log_brent','vix']])
y = d['log_inr']
beer = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags':6})
print(f"\nBEER REGRESSION:")
print(f"  R² = {beer.rsquared:.3f}   Adj R² = {beer.rsquared_adj:.3f}")
print(f"  {'variable':<16}{'coef':>9}{'p-val':>8}  interpretation")
interp = {
    'log_dxy':        'DXY↑1% → INR depreciates',
    'real_rate_diff':  'risk-premium channel',
    'fpi_pct_gdp':     'portfolio flow effect',
    'log_brent':       'oil pass-through',
    'vix':             'global risk aversion',
}
for v in ['log_dxy','real_rate_diff','fpi_pct_gdp','log_brent','vix']:
    sig = '***' if beer.pvalues[v]<0.01 else ('**' if beer.pvalues[v]<0.05 else ('*' if beer.pvalues[v]<0.1 else ''))
    print(f"  {v:<16}{beer.params[v]:>+9.4f}{beer.pvalues[v]:>8.4f}{sig:<4}{interp[v]}")

# ── 2) DXY dominance check ──────────────────────────────────────
beer_dxy = sm.OLS(y, sm.add_constant(d[['log_dxy']])).fit()
india_pct = ((beer.rsquared - beer_dxy.rsquared) / (1 - beer_dxy.rsquared)) * 100
print(f"\n  DXY-only R² = {beer_dxy.rsquared:.3f}")
print(f"  Full model R² = {beer.rsquared:.3f}")
print(f"  India-specific factors explain {india_pct:.0f}% of residual variance")

# ── 3) Cointegration test ───────────────────────────────────────
resid = beer.resid
adf = adfuller(resid, maxlag=12, autolag='AIC')
print(f"\nCOINTEGRATION (ADF on BEER residuals):")
print(f"  ADF stat = {adf[0]:.3f}  p = {adf[1]:.4f}  lags = {adf[2]}")
print(f"  {'✓ COINTEGRATED' if adf[1]<0.05 else '✗ NOT cointegrated'}")

# ── 4) BEER misalignment ───────────────────────────────────────
d['log_inr_fair'] = beer.fittedvalues
d['inr_fair_beer'] = np.exp(d['log_inr_fair'])
d['beer_misalign'] = (d['inr_usd'] / d['inr_fair_beer'] - 1) * 100

# ── 5) Episode validation ──────────────────────────────────────
episodes = {
    'GFC (Oct 2008)'          : '2008-10-01',
    'Taper (Aug 2013)'        : '2013-08-01',
    'COVID (Apr 2020)'        : '2020-04-01',
    'Fed tight (Oct 2022)'    : '2022-10-01',
    'Pre-war (Dec 2025)'      : '2025-12-01',
    'Oil shock (Mar 2026)'    : '2026-03-01',
}
print(f"\nEPISODE VALIDATION:")
print(f"  {'Episode':<28}{'INR act':>9}{'INR fair':>9}{'Misalign%':>10}")
for lbl, dt_str in episodes.items():
    dt = pd.Timestamp(dt_str)
    if dt in d.index:
        r = d.loc[dt]
        print(f"  {lbl:<28}{r['inr_usd']:>9.2f}{r['inr_fair_beer']:>9.2f}"
              f"{r['beer_misalign']:>10.1f}")
    else:
        print(f"  {lbl:<28}  (outside BEER sample)")

# ── 6) Regime matrix: Oil × DXY ────────────────────────────────
d['inr_ret'] = d['log_inr'].diff() * 100
brent_med = d['brent'].median(); dxy_med = d['log_dxy'].median()
d['oil_regime'] = np.where(d['brent']>brent_med, 'Hi', 'Lo')
d['dxy_regime'] = np.where(d['log_dxy']>dxy_med, 'Hi', 'Lo')
d['regime_4']   = d['oil_regime'] + '_' + d['dxy_regime']

print(f"\nOIL×DXY REGIME MATRIX (INR depreciation %/mo):")
print(f"  Brent median: ${brent_med:.0f}   DXY median: {np.exp(dxy_med):.0f}")
print(f"  {'Regime':<10}{'mean':>8}{'sd':>8}{'n':>6}")
for reg in ['Lo_Lo','Lo_Hi','Hi_Lo','Hi_Hi']:
    sub = d.loc[d['regime_4']==reg, 'inr_ret'].dropna()
    print(f"  {reg:<10}{sub.mean():>+8.2f}{sub.std():>8.2f}{len(sub):>6}")

# ── 7) Markov-switching 2-regime ─────────────────────────────────
ms_data = d['inr_ret'].dropna()
ms = MarkovRegression(ms_data, k_regimes=2, trend='c', switching_variance=True)
ms_fit = ms.fit(maxiter=500, em_iter=200)

# Identify regimes by volatility
sig0 = np.sqrt(ms_fit.params['sigma2[0]'])
sig1 = np.sqrt(ms_fit.params['sigma2[1]'])
if sig0 < sig1:
    calm, stress = 0, 1
else:
    calm, stress = 1, 0

mu_c = ms_fit.params[f'const[{calm}]'];  sig_c = np.sqrt(ms_fit.params[f'sigma2[{calm}]'])
mu_s = ms_fit.params[f'const[{stress}]']; sig_s = np.sqrt(ms_fit.params[f'sigma2[{stress}]'])
p_cc = ms_fit.params[f'p[{calm}->{calm}]']
p_sc = ms_fit.params[f'p[{stress}->{calm}]']   # prob stress→calm
dur_c = 1 / (1 - p_cc)
dur_s = 1 / p_sc     # duration of stress = 1/P(leave stress)

print(f"\nMARKOV-SWITCHING REGIMES:")
print(f"  Calm:   μ={mu_c:+.2f}%/mo  σ={sig_c:.2f}%  duration={dur_c:.1f}mo  p_stay={p_cc:.3f}")
print(f"  Stress: μ={mu_s:+.2f}%/mo  σ={sig_s:.2f}%  duration={dur_s:.1f}mo  p_stay={1-p_sc:.3f}")

# Smoothed probabilities
d['p_stress'] = ms_fit.smoothed_marginal_probabilities[stress].reindex(d.index)
print(f"\n  Regime probabilities — last 8 months:")
for dt in d.index[-8:]:
    if pd.notna(d.loc[dt,'p_stress']):
        ps = d.loc[dt,'p_stress']
        print(f"    {dt.strftime('%b %Y')}: P(stress)={ps:.2f}  "
              f"[{'█'*int(ps*20)+'░'*(20-int(ps*20))}]  "
              f"{'STRESS' if ps>0.5 else 'calm'}")

# ── 8) Save outputs ─────────────────────────────────────────────
# BEER output (monthly)
out_beer = d[['inr_usd','inr_fair_beer','beer_misalign','inr_ret','regime_4','p_stress']].copy()
out_beer.to_csv(PROCESSED+'phase4_beer_v2.csv')

# Coefficients
coefs = {v: float(beer.params[v]) for v in beer.params.index}
coefs['r_squared'] = float(beer.rsquared)
coefs['adf_p'] = float(adf[1])
with open(PROCESSED+'beer_coefs_v2.json','w') as f: json.dump(coefs, f, indent=2)

print(f"\n✓ Saved: phase4_beer_v2.csv ({out_beer.shape[0]} × {out_beer.shape[1]})")
print(f"         beer_coefs_v2.json")

# ── Cross-model summary ────────────────────────────────────────
print(f"\n{'='*70}")
print(f"FOUR-MODEL CONVERGENCE (latest available date)")
print(f"{'='*70}")

# Load Phase 2 + Phase 3 for comparison
p2 = pd.read_csv(PROCESSED+'phase2_equilibrium_v2.csv', index_col=0, parse_dates=True)
p2.index = pd.to_datetime(p2.index).to_period('M').to_timestamp()
p3 = pd.read_csv(PROCESSED+'phase3_feer_v2.csv', index_col=0, parse_dates=True)
p3.index = pd.to_datetime(p3.index).to_period('M').to_timestamp()

# REER (Phase 2) — latest
r2 = p2.iloc[-1]
print(f"\n  Phase 2 REER ({p2.index[-1].strftime('%b %Y')}):")
print(f"    Misalignment: {r2['misalignment_pct']:+.1f}%  Fair: {r2['inr_fair_reer']:.2f}")

# FEER (Phase 3) — latest quarter
r3 = p3.iloc[-1]
print(f"  Phase 3 FEER ({p3.index[-1].strftime('%b %Y')}):")
print(f"    Static:      {r3['feer_static_pct']:+.1f}%  Fair: {r3['inr_fair_static']:.2f}")
print(f"    Conditional: {r3['feer_cond_pct']:+.1f}%  Fair: {r3['inr_fair_cond']:.2f}")
print(f"    Fin. premium: {r3['financing_premium']:+.1f}pp")

# BEER (Phase 4) — latest
if len(d):
    r4 = d.iloc[-1]
    print(f"  Phase 4 BEER ({d.index[-1].strftime('%b %Y')}):")
    print(f"    Misalignment: {r4['beer_misalign']:+.1f}%  Fair: {r4['inr_fair_beer']:.2f}")
    print(f"    P(stress):    {r4['p_stress']:.2f}")

PHASE 4 — BEER EQUILIBRIUM + REGIME ENGINE
Sample: Feb 2001 → Mar 2026  (n=301)

BEER REGRESSION:
  R² = 0.852   Adj R² = 0.850
  variable             coef   p-val  interpretation
  log_dxy           +1.6874  0.0000*** DXY↑1% → INR depreciates
  real_rate_diff    +0.0108  0.0030*** risk-premium channel
  fpi_pct_gdp       -0.0011  0.8144    portfolio flow effect
  log_brent         +0.1474  0.0000*** oil pass-through
  vix               -0.0027  0.0333**  global risk aversion

  DXY-only R² = 0.746
  Full model R² = 0.852
  India-specific factors explain 42% of residual variance

COINTEGRATION (ADF on BEER residuals):
  ADF stat = -3.332  p = 0.0135  lags = 0
  ✓ COINTEGRATED

EPISODE VALIDATION:
  Episode                       INR act INR fair Misalign%
  GFC (Oct 2008)                  48.64    49.86      -2.5
  Taper (Aug 2013)                63.21    55.70      13.5
  COVID (Apr 2020)                76.24    61.98      23.0
  Fed tight (Oct 2022)            82.34    90.20      -8.7